In [ ]:
import requests, json, time, pathlib, urllib.parse, glob
import pandas as pd

APP_ID = "MNRWEFSS2Q"
API_KEY = "NGUzODljMDdlMzc5ODBkNzVhZjExMmZmYjExYWExNGY3NWZkZWMwODgxNTUzOGUxZWIxYmVkMGFiZmUxZjRiZHZhbGlkVW50aWw9MTc5MzczMTM5OCZ1c2VyVG9rZW49MTk0NjQ0MTEmZmlsdGVycz12aXNpYmxlX2J5JTNBKzE5NDY0NDEx"      # din request-ul cu Sold activ
INDEX = "Listing_sold_production"
URL = f"https://{APP_ID.lower()}-dsn.algolia.net/1/indexes/*/queries"
QS = {"x-algolia-agent": "Algolia for JavaScript (4.14.3); Browser"}

headers = {
    "x-algolia-application-id": APP_ID,
    "x-algolia-api-key": API_KEY,
    "Content-Type": "application/x-www-form-urlencoded",
    "Origin": "https://www.grailed.com",
    "Referer": "https://www.grailed.com/",
    "User-Agent": "Mozilla/5.0 (academic project)",
}

raw = pathlib.Path("raw/grailed_sold"); raw.mkdir(parents=True, exist_ok=True)

def fetch(query, page, lo=0, hi=1000000):
    p = {
        "query": query,
        "hitsPerPage": 40,
        "page": page,
        "numericFilters": json.dumps([f"sold_price>={lo}", f"sold_price<={hi}"]),
    }
    body = {"requests": [{"indexName": INDEX, "params": urllib.parse.urlencode(p)}]}
    r = requests.post(URL, params=QS, headers=headers, data=json.dumps(body), timeout=30)
    if r.status_code != 200:
        raise RuntimeError(f"{r.status_code}: {r.text[:300]}")
    return r.json()["results"][0]

def crawl(query, lo, hi):
    first = fetch(query, 0, lo, hi)
    n = first["nbHits"]
    if n == 0:
        return
    if n > 1000 and hi > lo:                 # bisecție pe preț
        mid = (lo + hi) // 2
        crawl(query, lo, mid)
        crawl(query, mid + 1, hi)
        return
    for page in range(min(first["nbPages"], 25)):
        res = first if page == 0 else fetch(query, page, lo, hi)
        name = f"{query.replace(' ','_')}_{lo}-{hi}_p{page}.json"
        (raw / name).write_text(json.dumps(res))
        time.sleep(2)

# 1) Test: inspectează un hit
# res = fetch("jordan 1", 0)
# print(res["nbHits"], res["nbPages"])
# h = res["hits"][0]
# print({k: h.get(k) for k in ["id","title","price_i","sold_price","sold_at","sold","designer_names","followerno","heat_f","strata"]})


for q in ["jordan 1"]:
    crawl(q, 0, 5000)


rows = []
for f in glob.glob("raw/grailed_sold/*.json"):
    for h in json.load(open(f))["hits"]:
        drops = h.get("price_drops") or []
        rows.append({
            "listing_id": h["id"], "platform": "grailed",
            "brand": h.get("designer_names"), "title": h.get("title"),
            "category": h.get("category"), "subcategory": h.get("category_path"),
            "size": h.get("size"), "condition": h.get("condition"),
            "color": h.get("color"), "country": h.get("location"),
            "sold_price": h.get("sold_price"),
            "last_asking_price": h.get("price_i"),
            "first_price": drops[0] if drops else h.get("price_i"),
            "n_price_drops": len(drops),
            "followers": h.get("followerno"),
            "heat": h.get("heat_f"), "strata": h.get("strata"),
            "photos": h.get("photo_count"),
            "listed_at": h.get("created_at"), "sold_at": h.get("sold_at"),
            "makeoffer": h.get("makeoffer"),
        })
df = pd.DataFrame(rows).drop_duplicates("listing_id")
df = df[df["sold_price"] > 0]
df.to_csv("grailed_sold.csv", index=False)
print(df.shape)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
pd.set_option("display.width", None)

print(df)


In [ ]:
print(df.head(10))

In [ ]:
df.to_csv("../data/grailed_sold.csv", index=False, sep=";", encoding="utf-8-sig")